# Session 2: shape sets the lift, lift costs drag
**A. Real-world problem.** Session 1 said a 1000 kg aeroplane at 50 m/s needs CL = 0.40.
Which wing section gives that, at what angle? And once we know the lift, how big is the
drag, and what is the best lift-to-drag ratio the aircraft can reach?

**Plan.** (1) the NACA 4-digit airfoil: camber line plus thickness, (2) thin-airfoil
theory: CL = 2 pi (alpha - alpha_L0), with alpha_L0 from the camber line, (3) the drag
polar of a whole aircraft, CD = CD0 + CL^2 / (pi e AR), and the best L/D.

**Model.** Low speed (no compressibility), attached flow (well below stall). A simplified
teaching model, not a design tool.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the NACA 4-digit airfoil
"NACA 2412" means: maximum camber m = 2 % of the chord, at p = 4 tenths of the chord from
the nose, maximum thickness t = 12 % of the chord. x runs from 0 (leading edge) to 1
(trailing edge) in units of the chord c.

* camber line: yc = m/p^2 (2 p x - x^2) for x < p, and m/(1-p)^2 ((1 - 2p) + 2 p x - x^2) after
* thickness: yt = 5 t (0.2969 sqrt(x) - 0.1260 x - 0.3516 x^2 + 0.2843 x^3 - 0.1036 x^4)
* the surfaces are the camber line plus and minus yt, measured at right angles to it.

In [ ]:
def parse_naca(code):
    """'2412' -> (m, p, t) as fractions of the chord."""
    if len(code) != 4 or not code.isdigit():
        raise ValueError("a NACA 4-digit code has exactly four digits, e.g. '2412'")
    m, p, t = int(code[0]) / 100, int(code[1]) / 10, int(code[2:]) / 100
    if m > 0 and not 0 < p < 1:
        raise ValueError("a cambered airfoil needs 1 <= second digit <= 9")
    if t <= 0:
        raise ValueError("thickness must be positive")
    return m, p, t


def camber_line(x, m, p):
    """Camber height yc(x) and slope dyc/dx for a NACA 4-digit camber line."""
    x = np.asarray(x, float)
    if m == 0:
        return np.zeros_like(x), np.zeros_like(x)
    front = x < p
    yc = np.where(front, m / p**2 * (2 * p * x - x**2), m / (1 - p) ** 2 * ((1 - 2 * p) + 2 * p * x - x**2))
    dyc = np.where(front, 2 * m / p**2 * (p - x), 2 * m / (1 - p) ** 2 * (p - x))
    return yc, dyc


def thickness(x, t):
    """Half-thickness yt(x) of the NACA 4-digit family (trailing edge slightly open)."""
    x = np.asarray(x, float)
    return 5 * t * (0.2969 * np.sqrt(x) - 0.1260 * x - 0.3516 * x**2 + 0.2843 * x**3 - 0.1036 * x**4)


def naca4(code, n=80):
    """Airfoil outline as (X, Y): from the trailing edge along the lower surface to the nose
    and back along the upper surface (clockwise), with points packed near both edges."""
    m, p, t = parse_naca(code)
    x = 0.5 * (1 - np.cos(np.linspace(0.0, np.pi, n + 1)))   # cosine spacing
    yc, dyc = camber_line(x, m, p)
    yt, th = thickness(x, t), np.arctan(dyc)
    xu, yu = x - yt * np.sin(th), yc + yt * np.cos(th)
    xl, yl = x + yt * np.sin(th), yc - yt * np.cos(th)
    return np.r_[xl[::-1], xu[1:]], np.r_[yl[::-1], yu[1:]]

## Step 2: thin-airfoil theory
Squash the airfoil to its camber line and replace it by a sheet of tiny whirls (vortices).
Requiring the flow to slide along the camber line and to leave the trailing edge smoothly
(the Kutta condition) gives a famous result:

    CL = 2 pi (alpha - alpha_L0)          (alpha in radians)

The slope 2 pi per radian (0.110 per degree) is the same for every thin airfoil. Only the
camber matters for alpha_L0, the angle at which the lift is zero:

    alpha_L0 = -(1/pi) * integral from 0 to pi of dyc/dx (cos theta - 1) d theta,
    with x = (1 - cos theta) / 2

A symmetric airfoil has alpha_L0 = 0. Camber makes alpha_L0 negative, so the wing lifts
even at zero angle. Thickness does not appear at all; Session 3 checks how good that is.

In [ ]:
def alpha_zero_lift(m, p, n=2000):
    """Zero-lift angle in degrees from thin-airfoil theory (midpoint rule in theta)."""
    theta = (np.arange(n) + 0.5) * np.pi / n
    x = 0.5 * (1 - np.cos(theta))
    _, dyc = camber_line(x, m, p)
    return float(np.degrees(-np.sum(dyc * (np.cos(theta) - 1.0)) * (np.pi / n) / np.pi))


def cl_thin(alpha_deg, code):
    """Lift coefficient from thin-airfoil theory for a NACA 4-digit code."""
    m, p, _ = parse_naca(code)
    return 2 * np.pi * np.radians(np.asarray(alpha_deg, float) - alpha_zero_lift(m, p))


def alpha_for_cl(cl, code):
    """Angle of attack in degrees that gives this CL (thin-airfoil theory)."""
    m, p, _ = parse_naca(code)
    return float(np.degrees(cl / (2 * np.pi)) + alpha_zero_lift(m, p))

## Step 3: the drag polar of a whole aircraft
A finite wing leaks air round its tips. The tip vortices tilt the lift backwards a little,
which is a drag that grows with the square of the lift: the induced drag. With aspect
ratio AR = span^2 / S and span efficiency e (about 0.7 to 0.9):

    CD = CD0 + k CL^2,     k = 1 / (pi e AR)

CD0 is the "parasite" drag of the whole aircraft at zero lift (skin friction, fuselage,
wheels). L/D = CL / CD is largest where the two parts are equal, CD0 = k CL^2, so

    CL* = sqrt(CD0 / k)        (L/D)max = 1 / (2 sqrt(CD0 k))

In [ ]:
def induced_factor(aspect_ratio, e=0.8):
    """k = 1 / (pi e AR) of the drag polar."""
    if aspect_ratio <= 0 or not 0 < e <= 1:
        raise ValueError("need AR > 0 and 0 < e <= 1")
    return 1.0 / (np.pi * e * aspect_ratio)


def cd_polar(cl, cd0, aspect_ratio, e=0.8):
    """Total drag coefficient of the aircraft at lift coefficient cl."""
    return cd0 + induced_factor(aspect_ratio, e) * np.asarray(cl, float) ** 2


def best_glide(cd0, aspect_ratio, e=0.8):
    """(CL*, best L/D) of the parabolic drag polar."""
    k = induced_factor(aspect_ratio, e)
    return float(np.sqrt(cd0 / k)), float(1.0 / (2.0 * np.sqrt(cd0 * k)))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)

    for code in ("0012", "2412", "4412"):
        m, p, _ = parse_naca(code)
        print(f"NACA {code}: alpha_L0 = {alpha_zero_lift(m, p):.3f} deg, CL(4 deg) = {cl_thin(4, code):.4f}, "
              f"alpha for CL 0.40 = {alpha_for_cl(0.40, code):.2f} deg")
    k = induced_factor(8, 0.8)
    cls, ld = best_glide(0.025, 8, 0.8)
    print(f"AR 8, e 0.8: k = {k:.5f}; CD0 0.025 -> CL* = {cls:.4f}, best L/D = {ld:.2f}")
    cd = cd_polar(0.40, 0.025, 8, 0.8)
    print(f"at CL 0.40: CD = {cd:.5f}, L/D = {0.40 / cd:.2f}")

    # Figure 1: three NACA airfoils, shape and camber line
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for code, col, off in (("0012", "#1f6feb", 0.0), ("2412", "#2a9d8f", -0.22), ("4412", "#d9534f", -0.44)):
        X, Y = naca4(code, 80)
        m, p, _ = parse_naca(code)
        xc = np.linspace(0, 1, 200)
        ax.fill(X, Y + off, color=col, alpha=0.25)
        ax.plot(X, Y + off, color=col, lw=1.5, label=f"NACA {code}")
        ax.plot(xc, camber_line(xc, m, p)[0] + off, "k--", lw=0.8)
    ax.set_aspect("equal")
    ax.set_xlabel("x / c (fraction of the chord)")
    ax.set_ylabel("y / c (shifted for clarity)")
    ax.set_title("NACA 4-digit airfoils: thickness wrapped round a camber line (dashed)")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "naca_shapes.png", dpi=120)

    # Figure 2: (left) thin-airfoil lift lines; (right) drag polar and L/D
    fig2, axes = plt.subplots(1, 2, figsize=(10, 4))
    al = np.linspace(-6, 10, 100)
    for code, col in (("0012", "#1f6feb"), ("2412", "#2a9d8f"), ("4412", "#d9534f")):
        axes[0].plot(al, cl_thin(al, code), color=col, label=f"NACA {code}")
        a0 = alpha_zero_lift(*parse_naca(code)[:2])
        axes[0].plot(a0, 0, "o", color=col)
    axes[0].axhline(0, color="k", lw=0.8)
    axes[0].axhline(0.40, color="k", ls=":", lw=1)
    axes[0].text(-5.8, 0.45, "CL = 0.40 (cruise, Session 1)", fontsize=8)
    axes[0].set_xlabel("angle of attack alpha (degrees)")
    axes[0].set_ylabel("lift coefficient CL (no unit)")
    axes[0].set_title("Thin-airfoil theory: same slope, camber shifts the line")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)
    cl = np.linspace(0.0, 1.4, 200)
    for ar, col in ((6, "#d9534f"), (8, "#2a9d8f"), (12, "#1f6feb")):
        axes[1].plot(cl, cl / cd_polar(cl, 0.025, ar, 0.8), color=col, label=f"AR = {ar}")
        c_s, ld_s = best_glide(0.025, ar, 0.8)
        axes[1].plot(c_s, ld_s, "o", color=col)
        axes[1].annotate(f"{ld_s:.1f}", (c_s, ld_s), xytext=(4, 4), textcoords="offset points", fontsize=8)
    axes[1].set_xlabel("lift coefficient CL (no unit)")
    axes[1].set_ylabel("lift-to-drag ratio L/D (no unit)")
    axes[1].set_title("CD = 0.025 + CL^2 / (pi 0.8 AR)")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)
    axes[1].set_ylim(0, 20)
    fig2.tight_layout()
    fig2.savefig(OUT / "lift_line_and_polar.png", dpi=120)

## Where this stops
Thin-airfoil theory ignores thickness and gives no pressure along the surface. Session 3
builds a panel method that computes the real speed and pressure all round any airfoil,
validates it, and adds a skin-friction estimate to get the airfoil's own lift and drag.

## Coding tasks
1. **Easy.** Change the code from '2412' to '2212', '2612' and '6412'. Predict which of the three changes moves alpha_L0 and in which direction, then run it.
2. **Stretch.** Write `thickness_ratio(code)` that computes the largest 2 yt from the outline `naca4(code)` and check it returns 0.12 for NACA 2412 (within 0.001).
3. **Explain.** Explain why gliders have long, thin wings while fighter jets have short ones.

## Assessment
- **A. Conceptual.** Two airfoils have the same thickness but one is symmetric and one is cambered. According to thin-airfoil theory, how do their lift lines differ, and which one needs the smaller angle to give CL = 0.40?
- **B. Numerical.** NACA 4412 has twice the camber of NACA 2412. Find its zero-lift angle and its CL at 2 degrees from thin-airfoil theory. Show the working.
- **C. Graph and simulation interpretation.** In the L/D figure, how do the best L/D and the CL where it occurs change when the aspect ratio rises from 6 to 12? Explain the shape of each curve.
- **D. Design and optimisation challenge.** Design: with CD0 = 0.025 and e = 0.8, what aspect ratio gives a best L/D of at least 15? For a 16 m2 wing, what span is that? Name two reasons not to choose a much larger AR.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("NACA 2412 means m = 0.02, p = 0.4, t = 0.12",
                   np.allclose(L.parse_naca("2412"), (0.02, 0.4, 0.12))))
    x = np.linspace(0, 1, 401)
    yc, dyc = L.camber_line(x, 0.02, 0.4)
    checks.append(("camber line peaks at height m = 0.02 at x = p = 0.4 with zero slope",
                   abs(yc.max() - 0.02) < 1e-12 and abs(x[np.argmax(yc)] - 0.4) < 1e-12 and abs(dyc[160]) < 1e-12))
    checks.append(("camber line starts and ends on the chord line", abs(yc[0]) < 1e-15 and abs(yc[-1]) < 1e-15))
    checks.append(("NACA 0012 is 12 % thick (largest 2 yt is 0.120 near x = 0.3)",
                   abs(2 * L.thickness(x, 0.12).max() - 0.12) < 5e-4 and abs(x[np.argmax(L.thickness(x, 0.12))] - 0.3) < 0.01))
    X, Y = L.naca4("0012", 60)
    checks.append(("symmetric outline: lower surface is the mirror of the upper", np.allclose(Y[:61][::-1], -Y[60:])))
    checks.append(("symmetric airfoil: zero-lift angle is 0", abs(L.alpha_zero_lift(0.0, 0.4)) < 1e-12))
    checks.append(("NACA 2412 zero-lift angle is -2.08 degrees (textbook value)", abs(L.alpha_zero_lift(0.02, 0.4) + 2.077) < 2e-3))
    checks.append(("zero-lift angle doubles with double camber (thin theory is linear)",
                   abs(L.alpha_zero_lift(0.04, 0.4) - 2 * L.alpha_zero_lift(0.02, 0.4)) < 1e-9))
    checks.append(("lift slope is 2 pi per radian (0.1097 per degree)",
                   abs((L.cl_thin(5, "2412") - L.cl_thin(4, "2412")) - 2 * np.pi * np.pi / 180) < 1e-12))
    checks.append(("worked example: NACA 2412 at 4 degrees gives CL = 0.666", abs(L.cl_thin(4, "2412") - 0.6664) < 1e-3))
    checks.append(("worked example: CL 0.40 on NACA 2412 needs 1.57 degrees",
                   abs(L.alpha_for_cl(0.40, "2412") - 1.57) < 0.01 and abs(L.cl_thin(L.alpha_for_cl(0.4, "2412"), "2412") - 0.4) < 1e-12))
    k = L.induced_factor(8, 0.8)
    checks.append(("induced-drag factor k = 1 / (pi 0.8 8) = 0.04974", abs(k - 0.04974) < 1e-5))
    cls, ld = L.best_glide(0.025, 8, 0.8)
    cl = np.linspace(0.05, 2, 20001)
    num = cl / L.cd_polar(cl, 0.025, 8, 0.8)
    checks.append(("best L/D 14.18 at CL* 0.709 and it matches a brute-force search",
                   abs(ld - 14.18) < 0.01 and abs(cls - 0.709) < 1e-3 and abs(num.max() - ld) < 1e-6 and abs(cl[np.argmax(num)] - cls) < 1e-3))
    checks.append(("at CL* the induced drag equals the parasite drag", abs(k * cls**2 - 0.025) < 1e-12))
    bad = 0
    for call in (lambda: L.parse_naca("241"), lambda: L.parse_naca("2012"), lambda: L.induced_factor(0, 0.8),
                 lambda: L.induced_factor(8, 1.2)):
        try:
            call()
        except ValueError:
            bad += 1
    checks.append(("invalid inputs raise ValueError", bad == 4))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
